# Exercise 4.1 — The C-F Identity (OTOC-Commutator Relation)

**Chapter 4: Scrambling Dynamics** &nbsp;|&nbsp; *Section 4.1: Out-of-Time-Order Correlators*

---

## Motivation

The **out-of-time-order correlator** (OTOC) $F(t) = \langle W^\dagger(t) V^\dagger W(t) V\rangle$ is the primary diagnostic of quantum information scrambling.  The C-F identity $C(t) = 2 - 2\,\mathrm{Re}\,F(t)$ connects it to the physically intuitive squared commutator.  This relation is the starting point for every scrambling analysis in this book — from Lyapunov growth to design verification.

## Preliminaries / Toolbox

Before diving into the solution, recall the following concepts:

**1. Out-of-Time-Order Correlators (OTOC):** A powerful metric for scrambling that tracks how an initially local operator $W_t = e^{iHt} W e^{-iHt}$ grows in time to fail to commute with another local operator $V$. $F(t) = \langle W_t^\dagger V^\dagger W_t V \rangle$.

**2. Squared Commutator:** $C(t) = \langle [W_t, V]^\dagger [W_t, V] \rangle$. For unitary $W,V$, it relates exactly to $F(t)$.

**3. Infinite Temperature Average:** $\langle O \rangle = \mathrm{Tr}(O)/D$, representing the infinite-temperature equilibrium state.

**4. Unitarity:** $W^\dagger W = I, V^\dagger V = I$.



## Exercise Statement

For unitary operators $W, V$ and the infinite-temperature average $\langle \cdot \rangle = \mathrm{Tr}(\cdot)/D$:

$$
F(t) = \langle W^\dagger(t) V^\dagger W(t) V \rangle, \qquad C(t) = \langle [W(t), V]^\dagger [W(t), V] \rangle.
$$

**(a)** Show $C(t) = 2 - 2\,\mathrm{Re}\,F(t)$.

**(b)** At $t=0$ with $[W, V] = 0$: verify $C(0) = 0$, $F(0) = 1$.  What are the values when $F \to -1$ and $F \to 0$?

**(c)** Explain why $F \to 0$ (not $-1$) at late times for Haar-random $W(t)$.

## Solution

### Part (a): Expanding the squared commutator

Drop the time argument for brevity.  Expand:

$$
[W, V]^\dagger [W, V] = (WV - VW)^\dagger(WV - VW).
$$

$$
= V^\dagger W^\dagger WV - V^\dagger W^\dagger VW - W^\dagger V^\dagger WV + W^\dagger V^\dagger VW.
$$

Using unitarity $W^\dagger W = I$ and $V^\dagger V = I$:

$$
= \underbrace{V^\dagger V}_{I} - V^\dagger W^\dagger VW - W^\dagger V^\dagger WV + \underbrace{W^\dagger W}_{I} = 2I - F^\dagger - F,
$$

where we identified $F = V^\dagger W^\dagger VW = W^\dagger(t) V^\dagger W(t) V$ (the OTOC kernel) and $F^\dagger = W^\dagger V^\dagger WV$.

Taking the trace average:

$$
\boxed{C(t) = 2 - \langle F \rangle - \langle F^\dagger \rangle = 2 - 2\,\mathrm{Re}\,F(t).}
$$

### Part (b): Boundary values

At $t = 0$, if $[W, V] = 0$:

$$
F(0) = \langle W^\dagger V^\dagger W V \rangle = \langle I \rangle = 1, \qquad C(0) = 0.
$$

The two extreme cases:

- $F \to -1$: $C = 2 - 2(-1) = 4$.  Maximum non-commutativity (anticommutation).
- $F \to 0$: $C = 2$.  This is the **Haar scrambled** value.

### Part (c): Why $F \to 0$ for Haar-random unitaries

When $W(t) = U^\dagger W U$ with $U$ drawn from the Haar measure, the OTOC becomes a 4th-order polynomial in $U$.  By the Weingarten calculus (specifically the $k=2$ formula), the average over $U$ of such a product factorizes into traces.  Since $W$ and $V$ are traceless Paulis, the leading contribution vanishes:

$$
\mathbb{E}_U[F] = \mathbb{E}_U[\langle U^\dagger W^\dagger U V^\dagger U^\dagger W U V \rangle] \to 0.
$$

There is no preferred phase alignment between $W(t)$ and $V$ in a fully scrambled system.  The OTOC $F(t)$ decays to zero, not $-1$, because scrambling **randomizes** the relative phase rather than flipping it.

---
## Symbolic Verification (SymPy)

In [ ]:
import sympy as sp

# C(t) = <[W_t, V]^dag [W_t, V]> for unitary W, V
# Expand: [W,V]^dag [W,V] = (WV - VW)^dag (WV - VW)
#       = V^dag W^dag W V + W^dag V^dag V W - V^dag W^dag V W - W^dag V^dag W V
#       = 2I - W^dag V^dag W V - V^dag W^dag V W
#       = 2I - F - F^dag  where F = W^dag(t) V^dag W(t) V
print('Commutator-Fidelity identity:')
print('  C(t) = <[W_t,V]^dag [W_t,V]>')
print('       = 2 - F(t) - F*(t)')
print('       = 2 - 2 Re[F(t)]')
print('  where F(t) = <W^dag(t) V^dag W(t) V> is the OTOC.')
print()
print('Properties:')
print('  t=0: F(0) = 1, C(0) = 0  (no scrambling)')
print('  t->inf: F->0 (Haar), C->2  (maximal scrambling)')

---
## Numerical Verification

In [ ]:
import numpy as np
from scipy.stats import unitary_group
from scipy.linalg import expm

np.random.seed(42)
sx = np.array([[0,1],[1,0]], dtype=complex)
sz = np.array([[1,0],[0,-1]], dtype=complex)
D = 2

# Part (a): Verify C = 2 − 2 Re(F) for random unitaries
print("Part (a): C(t) = 2 − 2 Re(F) verification")
for _ in range(5):
    U = unitary_group.rvs(D)
    Wt = U.conj().T @ sx @ U
    V = sz
    comm = Wt @ V - V @ Wt
    C_val = np.trace(comm.conj().T @ comm).real / D
    F_val = np.trace(Wt.conj().T @ V.conj().T @ Wt @ V) / D
    CF = 2 - 2*F_val.real
    print(f"  C = {C_val:.6f}, 2−2Re(F) = {CF:.6f}, match: {np.isclose(C_val, CF)}")
    assert np.isclose(C_val, CF)

# Part (c): F → 0 for Haar-random
F_vals = []
for _ in range(10000):
    U = unitary_group.rvs(D)
    Wt = U.conj().T @ sx @ U
    F_vals.append(np.trace(Wt.conj().T @ sz.conj().T @ Wt @ sz) / D)

print(f"\nPart (c): E[F] = {np.mean(F_vals):.4f} (expected 0)")
print(f"C-F identity and Haar scrambling confirmed. ✓")